# System Modeling


<span id="실행-준비"></span>

## Execution setup

The Python option uses python-control. If the module is missing, run `%pip install control==0.10.2` in a Colab or Jupyter code cell.

<span id="학습-목표"></span>

## Learning objectives

Use the model to determine whether changing mass or resistance affects the transient response, the steady-state value, or both.

<span id="cruise-모델과-단위"></span>

## Cruise model and units

<span id="시작-조건"></span>

### Starting conditions

$m=1000$ kg, $b=50$ N·s/m, force 500 N. The state and output are velocity \[m/s\].

<span id="진행-방법"></span>

### Procedure

1.  Build a model using $A=-b/m$, $B=1/m$, $C=1$, and $D=0$, and plot the response over $0\le t\le150$ s to a constant force of 500 N.
2.  Change only $m\in\{500,1000,1500\}$ and apply the same force. Substitute $\dot v=0$ into $m\dot v+bv=500$ to obtain steady-state velocity, and match the coefficients of $\dot v+(b/m)v=500/m$ to the standard 1st-order form to obtain the time constant.
3.  For each mass, record the calculated steady-state velocity alongside the final simulation sample $v(150)$. Find when the curve reaches approximately 63.2% of the calculated steady-state velocity, and compare this time with the calculated time constant.

<span id="남길-결과"></span>

### Results to retain

- A table of 3 mass conditions, 1 velocity plot, and a brief explanation of the properties changed by mass.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
m=1000.; b=50.; force=500.
G=ctl.ss([[-b/m]],[[1/m]],[[1]],[[0]])
t=np.linspace(0,150,1501)
t,v=ctl.step_response(force*G,t)
plt.plot(t,v); plt.xlabel('Time [s]'); plt.ylabel('Speed [m/s]'); plt.grid()
# TODO: Compare the 3 mass conditions with the values predicted by the equations.


<span id="결과-기록자가-점검"></span>

## Record results and self-check

**m \[kg\]:**

**Predicted final velocity \[m/s\]:**

**Measured velocity \[m/s\]:**

**Time Constant \[s\]:**

<span id="확인할-점"></span>

### Points to check

- Did you avoid multiplying by the input force 500 twice?
- For fixed b, does changing mass affect the time constant rather than the steady-state velocity?

<span id="motor-모델의-계수-비교"></span>

## Comparing motor-model coefficients

<span id="시작-조건-1"></span>

### Starting conditions

$J=.01$, $b=.1$, $K=.01$, $L=.5$. Compare $R\in\{.5,1,2\}$ Ω with an input of 1 V.

<span id="진행-방법-1"></span>

### Procedure

1.  Construct $G(s)=\frac K{(Js+b)(Ls+R)+K^2}$ and calculate its response over $0\le t\le 10$ s.
2.  Tabulate the poles, DC gain, and final output for each R.
3.  Explain which terms in the equation change with electrical resistance and inertia.

<span id="남길-결과-1"></span>

### Results to retain

- 3 response curves and 1 coefficient table.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ctl
s = ctl.tf('s')
J=.01; b=.1; K=.01; L=.5
for R in [.5,1.,2.]:
    G=K/((J*s+b)*(L*s+R)+K**2)
    print(R,ctl.poles(G),ctl.dcgain(G))
# TODO: Add responses on the same time axis.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**R \[Ω\]:**

**Pole:**

**DC Gain:**

**Final velocity \[rad/s\]:**

<span id="확인할-점-1"></span>

### Points to check

- Did you use the same J, b, K, L, and input for all conditions?
- Did you compare the long-term output with 1 V×DC gain?

<span id="측정한-응답에서-모델-추정하기"></span>

## Estimating a model from a measured response

<span id="시작-조건-2"></span>

### Starting conditions

Synthetic sensor step record: $y=1.8(1-e^{-t/3})+\text{noise}$. $0\le t\le 15$ s, 151 points, noise standard deviation .03, `seed=7`.

<span id="진행-방법-2"></span>

### Procedure

1.  Use the supplied measurement array as the target values for `curve_fit`, and estimate $K,\tau$ of $K(1-e^{-t/\tau})$.
2.  Apply the same estimation separately to the full $0$–$15$ s record and the first $0$–$3$ s. Tabulate the sample count and estimated $K,\tau$ for each interval.
3.  Plot each fitted curve on the same time axis as the measurements, and plot residuals as measured minus estimated values. Based on the final values and residual patterns, choose one time interval where you would like more measurements and explain why.

<span id="남길-결과-2"></span>

### Results to retain

- A table of estimates for both measurement intervals and response and residual plots.

In [ ]:
from scipy.optimize import curve_fit
rng=np.random.default_rng(7); tm=np.linspace(0,15,151)
measured=1.8*(1-np.exp(-tm/3))+rng.normal(0,.03,len(tm))
def response(t,gain,tau):return gain*(1-np.exp(-t/tau))
for end in [3,15]:
    select=tm<=end
    parameters,_=curve_fit(response,tm[select],measured[select],p0=[1,2],bounds=(0,np.inf))
    print(end,parameters)
# TODO: Compare curves and residuals on the same time axis.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Recording interval \[s\]:**

**Estimated gain:**

**Estimated time constant \[s\]:**

**RMSE:**

<span id="확인할-점-2"></span>

### Points to check

- Did you estimate the synthetic data as if their generating settings were unknown, as with actual measurements?
- Did you avoid assessing uncertainty from a short record using residual size alone?

<span id="공식-예제데이터"></span>

## Official examples and data

- [SciPy · curve_fit](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html)